In [1]:
import numpy as np
import pandas as pd

# =========================================================
# Validation Metrics Toolkit
# Supports:
# 1) Direct inputs: y_true, pred_mean, pred_q_low, pred_q_high
# 2) MC samples: y_true, pred_samples
# =========================================================

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def pinball_loss(y_true, y_pred_quantile, tau):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred_quantile = np.asarray(y_pred_quantile, dtype=np.float64).reshape(-1)
    diff = y_true - y_pred_quantile
    return np.mean(np.maximum(tau * diff, (tau - 1) * diff))

def average_pinball_loss(y_true, pred_q_low, pred_q_high, low_tau=0.05, high_tau=0.95):
    low_loss = pinball_loss(y_true, pred_q_low, low_tau)
    high_loss = pinball_loss(y_true, pred_q_high, high_tau)
    return 0.5 * (low_loss + high_loss)

def coverage(y_true, pred_q_low, pred_q_high):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    pred_q_low = np.asarray(pred_q_low, dtype=np.float64).reshape(-1)
    pred_q_high = np.asarray(pred_q_high, dtype=np.float64).reshape(-1)
    inside = (y_true >= pred_q_low) & (y_true <= pred_q_high)
    return np.mean(inside)

def interval_width(pred_q_low, pred_q_high):
    pred_q_low = np.asarray(pred_q_low, dtype=np.float64).reshape(-1)
    pred_q_high = np.asarray(pred_q_high, dtype=np.float64).reshape(-1)
    return np.mean(pred_q_high - pred_q_low)

def interval_score(y_true, pred_q_low, pred_q_high, alpha=0.10):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    pred_q_low = np.asarray(pred_q_low, dtype=np.float64).reshape(-1)
    pred_q_high = np.asarray(pred_q_high, dtype=np.float64).reshape(-1)

    width = pred_q_high - pred_q_low
    below_penalty = (2.0 / alpha) * (pred_q_low - y_true) * (y_true < pred_q_low)
    above_penalty = (2.0 / alpha) * (y_true - pred_q_high) * (y_true > pred_q_high)

    return np.mean(width + below_penalty + above_penalty)

def summarize_from_samples(pred_samples, low_tau=0.05, high_tau=0.95):
    """
    pred_samples shape: [num_samples, N] or [S, N]
    """
    pred_samples = np.asarray(pred_samples, dtype=np.float64)
    if pred_samples.ndim != 2:
        raise ValueError("pred_samples must have shape [num_samples, N].")

    pred_mean = pred_samples.mean(axis=0)
    pred_q_low = np.quantile(pred_samples, low_tau, axis=0)
    pred_q_high = np.quantile(pred_samples, high_tau, axis=0)
    return pred_mean, pred_q_low, pred_q_high

def evaluate_validation_metrics(
    y_true,
    pred_mean=None,
    pred_q_low=None,
    pred_q_high=None,
    pred_samples=None,
    low_tau=0.05,
    high_tau=0.95,
    alpha=0.10
):
    """
    Either provide:
      - y_true, pred_mean, pred_q_low, pred_q_high
    or:
      - y_true, pred_samples

    Returns a dictionary of metrics.
    """
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)

    if pred_samples is not None:
        pred_mean, pred_q_low, pred_q_high = summarize_from_samples(
            pred_samples,
            low_tau=low_tau,
            high_tau=high_tau
        )
    else:
        if pred_mean is None or pred_q_low is None or pred_q_high is None:
            raise ValueError(
                "You must provide either pred_samples, or all of pred_mean / pred_q_low / pred_q_high."
            )
        pred_mean = np.asarray(pred_mean, dtype=np.float64).reshape(-1)
        pred_q_low = np.asarray(pred_q_low, dtype=np.float64).reshape(-1)
        pred_q_high = np.asarray(pred_q_high, dtype=np.float64).reshape(-1)

    if not (len(y_true) == len(pred_mean) == len(pred_q_low) == len(pred_q_high)):
        raise ValueError("All input arrays must have the same length.")

    results = {
        "MAE": mae(y_true, pred_mean),
        "RMSE": rmse(y_true, pred_mean),
        "Pinball Loss": average_pinball_loss(
            y_true,
            pred_q_low,
            pred_q_high,
            low_tau=low_tau,
            high_tau=high_tau
        ),
        "Coverage": coverage(y_true, pred_q_low, pred_q_high),
        "Interval Width": interval_width(pred_q_low, pred_q_high),
        "Interval Score": interval_score(y_true, pred_q_low, pred_q_high, alpha=alpha)
    }
    return results

def evaluate_by_group(
    y_true,
    group_ids,
    pred_mean=None,
    pred_q_low=None,
    pred_q_high=None,
    pred_samples=None,
    low_tau=0.05,
    high_tau=0.95,
    alpha=0.10
):
    """
    Evaluate metrics separately for each group.
    group_ids: item ids / store ids / any grouping labels
    pred_samples: shape [S, N] if provided
    """
    y_true = np.asarray(y_true)
    group_ids = np.asarray(group_ids)

    if pred_samples is not None:
        pred_samples = np.asarray(pred_samples)
        if pred_samples.ndim != 2:
            raise ValueError("pred_samples must have shape [num_samples, N].")
        if pred_samples.shape[1] != len(y_true):
            raise ValueError("pred_samples second dimension must match len(y_true).")

    if pred_mean is not None:
        pred_mean = np.asarray(pred_mean)
    if pred_q_low is not None:
        pred_q_low = np.asarray(pred_q_low)
    if pred_q_high is not None:
        pred_q_high = np.asarray(pred_q_high)

    rows = []
    for gid in np.unique(group_ids):
        mask = group_ids == gid

        if pred_samples is not None:
            group_results = evaluate_validation_metrics(
                y_true=y_true[mask],
                pred_samples=pred_samples[:, mask],
                low_tau=low_tau,
                high_tau=high_tau,
                alpha=alpha
            )
        else:
            group_results = evaluate_validation_metrics(
                y_true=y_true[mask],
                pred_mean=pred_mean[mask],
                pred_q_low=pred_q_low[mask],
                pred_q_high=pred_q_high[mask],
                low_tau=low_tau,
                high_tau=high_tau,
                alpha=alpha
            )

        row = {"group_id": gid}
        row.update(group_results)
        rows.append(row)

    return pd.DataFrame(rows)

def print_metrics(results, title="Validation Metrics"):
    print(f"\n{title}")
    print("-" * len(title))
    for k, v in results.items():
        print(f"{k}: {v:.6f}")

def metrics_dataframe(level_name, results_dict):
    row = {"Level": level_name}
    row.update(results_dict)
    return pd.DataFrame([row])


# =========================================================
# Example usage 1:
# Direct arrays
# =========================================================
# results = evaluate_validation_metrics(
#     y_true=y_true,
#     pred_mean=pred_mean,
#     pred_q_low=pred_q05,
#     pred_q_high=pred_q95,
#     low_tau=0.05,
#     high_tau=0.95,
#     alpha=0.10
# )
# print_metrics(results)


# =========================================================
# Example usage 2:
# From MC samples
# pred_samples shape = [num_samples, N]
# =========================================================
# results = evaluate_validation_metrics(
#     y_true=y_true,
#     pred_samples=pred_samples,
#     low_tau=0.05,
#     high_tau=0.95,
#     alpha=0.10
# )
# print_metrics(results)


# =========================================================
# Example usage 3:
# If your notebook already has these variables:
# detail["y_item"], detail["item_pred"], detail["item_q05"], detail["item_q95"]
# detail["y_store"], detail["store_pred"], detail["store_q05"], detail["store_q95"]
# =========================================================
# item_results = evaluate_validation_metrics(
#     y_true=detail["y_item"],
#     pred_mean=detail["item_pred"],
#     pred_q_low=detail["item_q05"],
#     pred_q_high=detail["item_q95"]
# )
#
# store_results = evaluate_validation_metrics(
#     y_true=detail["y_store"],
#     pred_mean=detail["store_pred"],
#     pred_q_low=detail["store_q05"],
#     pred_q_high=detail["store_q95"]
# )
#
# final_metrics_df = pd.concat([
#     metrics_dataframe("Item", item_results),
#     metrics_dataframe("Store", store_results)
# ], ignore_index=True)
#
# print_metrics(item_results, title="Item-level Validation Metrics")
# print_metrics(store_results, title="Store-level Validation Metrics")
# display(final_metrics_df)


# =========================================================
# Example usage 4:
# Group-wise evaluation
# =========================================================
# group_df = evaluate_by_group(
#     y_true=detail["y_item"],
#     pred_mean=detail["item_pred"],
#     pred_q_low=detail["item_q05"],
#     pred_q_high=detail["item_q95"],
#     group_ids=detail["item_ids"]
# )
# display(group_df.head())